# L1 SCENIC AUC differential analysis (wilcoxon)
Read AUC matrices from `/data1st1/junyi/output/sn0827L1/scenic_auc/`, attach cell metadata,
run wilcoxon per region x L1 celltype (status vs CON, split by sex), and save combined results.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import glob
import os

import anndata as ad
import numpy as np
from scipy import sparse
import pandas as pd
import scanpy as sc

In [2]:
input_dir = "/data1st1/junyi/output/sn0827L1/scenic_auc/"
out_dir = "/data2/junyi/stg0901/scenic_wilcoxon_L1/"
h5ad_dir = "/data2/junyi/stg0901/scenic_auc_L1_h5ad/"
meta_file = "/data2st2/junyi/output/stg1028/combined_ALL_meta.csv"

os.makedirs(out_dir, exist_ok=True)
files = sorted(glob.glob(os.path.join(input_dir, "auc*.tsv")))
print(f"{len(files)} auc files")
files[:5]

68 auc files


['/data1st1/junyi/output/sn0827L1/scenic_auc/auc_AMY_Astrocyte.tsv',
 '/data1st1/junyi/output/sn0827L1/scenic_auc/auc_AMY_Epen.tsv',
 '/data1st1/junyi/output/sn0827L1/scenic_auc/auc_AMY_GABA.tsv',
 '/data1st1/junyi/output/sn0827L1/scenic_auc/auc_AMY_Glut.tsv',
 '/data1st1/junyi/output/sn0827L1/scenic_auc/auc_AMY_Immune.tsv']

In [3]:
# cell metadata indexed by barcode: status, sex, region, celltype.L1, celltype.L2, Neurotransmitter_celltype
df_meta = pd.read_csv(meta_file, index_col=0)
df_meta.drop_duplicates(inplace=True)
if 'predicted_doublets' in df_meta.columns:
    df_meta['predicted_doublets'] = (
        df_meta['predicted_doublets'].replace({'True': True, 'False': False}).astype('boolean')
    )
print(df_meta.shape)

(1889112, 68)


In [4]:
# build (or load cached) combined AnnData of all region x L1 celltype AUC matrices
dict_pallium = {
    'PFC': 'Pallium', 'HPF': 'Pallium', 'STR': 'Subpallium',
    'AMY': 'Subpallium', 'TH': 'TH', 'HY': 'HY', 'MB': 'MB',
}

if os.path.exists(os.path.join(h5ad_dir, "combined_auc_all.h5ad")):
    adata_combined = sc.read_h5ad(os.path.join(h5ad_dir, "combined_auc_all.h5ad"))
else:
    adata_list = []
    for file in files:
        try:
            adata = sc.read_csv(file, delimiter="\t")
            missing_in_meta = set(adata.obs_names) - set(df_meta.index)
            if missing_in_meta:
                print("obs IDs missing in df_meta:", list(missing_in_meta)[:10])
                raise ValueError("adata.obs_names not fully found in df_meta.index")
            adata.obs = df_meta.loc[adata.obs_names]
            adata_list.append(adata)
        except Exception as e:
            print(f"Error processing file {file}: {e}")
            continue

    adata_combined = ad.concat(adata_list, axis=0, label='dataset', index_unique='-', join='outer')

    # h5ad cannot store NaN
    if sparse.issparse(adata_combined.X):
        adata_combined.X = adata_combined.X.tocsr(copy=True)
        adata_combined.X.data = np.nan_to_num(adata_combined.X.data, nan=0.0)
    else:
        adata_combined.X = np.nan_to_num(adata_combined.X, nan=0.0)

    # numeric-string columns -> float, everything else -> str, so h5ad writing works
    for col in adata_combined.obs.columns:
        if adata_combined.obs[col].dtype == object:
            try:
                adata_combined.obs[col] = adata_combined.obs[col].astype(float)
            except (ValueError, TypeError):
                adata_combined.obs[col] = adata_combined.obs[col].astype(str)

    # Region_NT: for non-NN cells use Region_L1type, otherwise the L1 type alone
    mask = adata_combined.obs['Neurotransmitter_celltype'].ne('NN')
    adata_combined.obs['Region_NT'] = adata_combined.obs['celltype.L1'].astype(str)
    adata_combined.obs.loc[mask, 'Region_NT'] = (
        adata_combined.obs.loc[mask, 'region'].astype(str) + '_' +
        adata_combined.obs.loc[mask, 'celltype.L1'].astype(str)
    )
    adata_combined.obs['pallium'] = adata_combined.obs['region'].map(dict_pallium)
    adata_combined.obs['pallium_NT'] = adata_combined.obs['celltype.L1'].astype(str)
    adata_combined.obs.loc[mask, 'pallium_NT'] = (
        adata_combined.obs.loc[mask, 'pallium'].astype(str) + '_' +
        adata_combined.obs.loc[mask, 'celltype.L1'].astype(str)
    )

    os.makedirs(h5ad_dir, exist_ok=True)
    adata_combined.write_h5ad(os.path.join(h5ad_dir, "combined_auc_all.h5ad"))

print(adata_combined.shape)
adata_combined.obs['celltype.L1'].value_counts()

(1338788, 966)


Glut         650910
GABA         391274
Oligo        125330
Astrocyte    102286
OPC           36961
Microglia      9006
Epen           7568
Vascular       5859
Chol           3552
Dopa           2590
Hist           1618
Sero           1407
Immune          427
Name: celltype.L1, dtype: int64

In [5]:
h5ad_dir

'/data2/junyi/stg0901/scenic_auc_L1_h5ad/'

In [6]:
# wilcoxon on AUC per region x L1 celltype file
# M: SUS / RES / CSDS / CSRES vs CON   |   F: SUS / RES vs CON
for file in files:
    name = os.path.basename(file).replace(".tsv", "")
    try:
        adata = sc.read_csv(file, delimiter="\t")
        if adata.n_obs == 0:
            print(f"skip {name}: no cells")
            continue
        missing = set(adata.obs_names) - set(df_meta.index)
        if missing:
            print(f"skip {name}: {len(missing)} barcodes missing in meta")
            continue
        adata.obs = df_meta.loc[adata.obs_names]

        df_result = pd.DataFrame()
        for sex in ['M', 'F']:
            for condition in ['SUS', 'RES', 'CSDS', 'CSRES']:
                if sex == 'F' and condition in ['CSRES', 'CSDS']:
                    continue
                adata_subset = adata[
                    (adata.obs.status.isin([condition, 'CON'])) & (adata.obs.sex == sex)
                ]
                if adata_subset.n_obs < 3:
                    print(f"skip {name} {condition}_{sex}: <3 cells")
                    continue
                try:
                    sc.tl.rank_genes_groups(adata_subset, groupby='status', method='wilcoxon', pts=True)
                except Exception as e:
                    print(f"fail {name} {condition}_{sex}: {e}")
                    continue
                df_deg = sc.get.rank_genes_groups_df(adata_subset, group=condition)
                df_deg['comparison'] = f'{condition}_{sex}'
                df_deg['status'] = condition
                df_deg['sex'] = sex
                for feature in ['region', 'celltype.L1', 'Neurotransmitter_celltype']:
                    if feature in adata_subset.obs.columns:
                        df_deg[feature] = adata_subset.obs[feature].astype(str).values[0]
                df_deg['Direction'] = np.where(df_deg['logfoldchanges'] > 0, 'Up', 'Down')
                df_deg.rename(columns={"names": "TF", "logfoldchanges": "log2FC",
                                       "pvals_adj": "FDR", "pvals": "pval"}, inplace=True)
                # one csv per comparison
                df_deg.to_csv(os.path.join(out_dir, f"{name}.{condition}_{sex}.csv"), index=False)
                df_result = pd.concat([df_result, df_deg], axis=0)

        # all comparisons of this file combined
        df_result.to_csv(os.path.join(out_dir, f"{name}.wilcoxon.csv"), index=False)
        print(f"done {name}: {adata.n_obs} cells, {len(df_result)} rows")
    except Exception as e:
        print(f"error {file}: {e}")
        continue

done auc_AMY_Astrocyte: 14888 cells, 1086 rows
done auc_AMY_Epen: 284 cells, 1680 rows
done auc_AMY_GABA: 71002 cells, 768 rows
done auc_AMY_Glut: 110968 cells, 654 rows
done auc_AMY_Immune: 32 cells, 1266 rows
done auc_AMY_Microglia: 1051 cells, 1698 rows
done auc_AMY_OPC: 9312 cells, 1314 rows
done auc_AMY_Oligo: 5042 cells, 1500 rows
done auc_AMY_Vascular: 973 cells, 1506 rows
done auc_HPF_Astrocyte: 7196 cells, 1242 rows
done auc_HPF_Epen: 1671 cells, 1518 rows
done auc_HPF_GABA: 16955 cells, 1314 rows
done auc_HPF_Glut: 154958 cells, 768 rows
done auc_HPF_Immune: 31 cells, 1326 rows
done auc_HPF_Microglia: 1238 cells, 1776 rows
done auc_HPF_OPC: 4411 cells, 1572 rows
done auc_HPF_Oligo: 12112 cells, 1176 rows
done auc_HPF_Vascular: 473 cells, 1470 rows
done auc_HY_Astrocyte: 22583 cells, 678 rows
done auc_HY_Chol: 977 cells, 1638 rows
done auc_HY_Epen: 3062 cells, 1290 rows
done auc_HY_GABA: 79407 cells, 852 rows
done auc_HY_Glut: 63895 cells, 864 rows
done auc_HY_Hist: 1618 cells

In [7]:
# combine all per-file results
files_w = sorted(glob.glob(os.path.join(out_dir, "*.wilcoxon.csv")))
df_deg_all = pd.concat([pd.read_csv(f) for f in files_w], axis=0, ignore_index=True)
df_deg_all.drop_duplicates(inplace=True)
print(df_deg_all.shape)
df_deg_all.to_csv(os.path.join(out_dir, "combined_wilcoxon_DEGs.csv"), index=False)

df_deg_sig = df_deg_all[df_deg_all['FDR'] < 0.05]
df_deg_sig.to_csv(os.path.join(out_dir, "combined_wilcoxon_DEGs_FDR05.csv"), index=False)
print(df_deg_sig.shape)

(71240, 14)
(20390, 14)


In [8]:
out_dir

'/data2/junyi/stg0901/scenic_wilcoxon_L1/'